# 01 — Fraud Signal Discovery

## Question

How does observed fraud exposure vary across the transaction environment?

This notebook audits the IEEE-CIS training data and examines whether fraud patterns differ across time, product categories, card characteristics, identity coverage, and transaction value.

The analysis is descriptive. Product categories are anonymized, identity information is only partially observed, and `TransactionDT` represents relative rather than calendar time.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/PROJECTS/fraud-friction")

RAW = BASE / "data" / "raw"
PROCESSED = BASE / "data" / "processed"

transaction_path = RAW / "train_transaction.csv"
identity_path = RAW / "train_identity.csv"

print(transaction_path.exists())
print(identity_path.exists())

In [ ]:
transactions = pd.read_csv(transaction_path)
identity = pd.read_csv(identity_path)

print("Transactions:", transactions.shape)
print("Identity:", identity.shape)

## 1. Transaction audit

Start with the analytical grain, fraud target, transaction amount, and identity coverage before comparing fraud patterns.


In [ ]:
print("=== SHAPE ===")
print("Transactions:", transactions.shape)
print("Identity:", identity.shape)

print("\n=== TRANSACTION ID CHECK ===")
print("Transaction rows:", len(transactions))
print("Unique TransactionIDs:", transactions["TransactionID"].nunique())
print("Duplicate TransactionIDs:", transactions["TransactionID"].duplicated().sum())

print("\n=== IDENTITY ID CHECK ===")
print("Identity rows:", len(identity))
print("Unique TransactionIDs:", identity["TransactionID"].nunique())
print("Duplicate TransactionIDs:", identity["TransactionID"].duplicated().sum())

print("\n=== IDENTITY COVERAGE ===")
identity_coverage = transactions["TransactionID"].isin(
    identity["TransactionID"]
).mean()

print(f"With identity: {identity_coverage:.2%}")
print(f"Without identity: {1 - identity_coverage:.2%}")

In [ ]:
fraud_summary = (
    transactions["isFraud"]
    .value_counts(dropna=False)
    .rename_axis("isFraud")
    .reset_index(name="transactions")
)

fraud_summary["percent"] = (
    fraud_summary["transactions"] / len(transactions) * 100
)

display(fraud_summary)

print(f"Overall fraud rate: {transactions['isFraud'].mean():.4%}")

In [ ]:
display(
    transactions["TransactionAmt"].describe(
        percentiles=[.01, .05, .25, .50, .75, .95, .99]
    )
)

amount_by_fraud = (
    transactions
    .groupby("isFraud")
    ["TransactionAmt"]
    .agg(["count", "mean", "median", "sum"])
)

display(amount_by_fraud)

### Finding 1

The transaction table has a unique transaction-level key and an observed fraud label, but fraud is a small minority of the data.

Identity information is available for only part of the transaction population, so identity-based comparisons should be treated as subset analysis rather than evidence about all transactions.

Transaction amounts are also strongly right-skewed, so count-based and value-based fraud exposure should be examined separately.


## 2. Does fraud risk change over time?

`TransactionDT` is converted into relative days and weeks so fraud exposure can be compared across the observed transaction sequence without assigning unsupported calendar dates.


In [ ]:
SECONDS_PER_DAY = 86400

min_dt = transactions["TransactionDT"].min()

transactions["RelativeDay"] = (
    (transactions["TransactionDT"] - min_dt) // SECONDS_PER_DAY
).astype(int) + 1

transactions["RelativeWeek"] = (
    (transactions["RelativeDay"] - 1) // 7
).astype(int) + 1

print("TransactionDT:")
display(transactions["TransactionDT"].describe())

print(
    f"Relative Day: {transactions['RelativeDay'].min()} "
    f"to {transactions['RelativeDay'].max()}"
)

print(
    f"Relative Week: {transactions['RelativeWeek'].min()} "
    f"to {transactions['RelativeWeek'].max()}"
)

In [ ]:
weekly = (
    transactions
    .groupby("RelativeWeek")
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum"),
        total_value=("TransactionAmt", "sum")
    )
    .reset_index()
)

weekly["fraud_rate"] = (
    weekly["fraud_transactions"] /
    weekly["transactions"]
)

display(weekly)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(
    weekly["RelativeWeek"],
    weekly["fraud_rate"],
    marker="o"
)

plt.xlabel("Relative Week")
plt.ylabel("Fraud Rate")
plt.title("Fraud Rate Over Relative Time")
plt.show()

### Finding 2

Fraud exposure changes across the observed relative weeks rather than remaining constant over time.

Because the source does not provide a real calendar timestamp, these changes cannot be tied to specific dates or external events. They do show that the portfolio-level fraud rate hides temporal variation.


## 3. What does the feature structure look like?

The transaction table contains several feature families with different levels of missingness and interpretability.

Before using them as analytical signals, I review the schema, missingness, and cardinality rather than treating every available variable as equally informative.


In [ ]:
column_families = {
    "core": [
        "TransactionID", "TransactionDT", "TransactionAmt",
        "ProductCD", "isFraud"
    ],
    "card": [c for c in transactions.columns if c.startswith("card")],
    "address": [c for c in transactions.columns if c.startswith("addr")],
    "email": ["P_emaildomain", "R_emaildomain"],
    "C": [c for c in transactions.columns if c.startswith("C") and c[1:].isdigit()],
    "D": [c for c in transactions.columns if c.startswith("D") and c[1:].isdigit()],
    "M": [c for c in transactions.columns if c.startswith("M") and c[1:].isdigit()],
    "V": [c for c in transactions.columns if c.startswith("V") and c[1:].isdigit()],
}

for family, cols in column_families.items():
    print(f"{family:10s}: {len(cols):3d} columns")
    print(cols[:15])
    print()

In [ ]:
profile = pd.DataFrame({
    "dtype": transactions.dtypes.astype(str),
    "missing_pct": transactions.isna().mean() * 100,
    "unique_values": transactions.nunique(dropna=True)
})

profile = profile.sort_values(
    ["missing_pct", "unique_values"],
    ascending=[False, False]
)

display(profile.head(50))

In [ ]:
interpretable_cols = [
    "TransactionAmt",
    "ProductCD",
    "card1", "card2", "card3", "card4", "card5", "card6",
    "addr1", "addr2",
    "P_emaildomain", "R_emaildomain",
    "M1", "M2", "M3", "M4", "M5", "M6", "M7", "M8", "M9"
]

display(profile.loc[interpretable_cols])

### Finding 3

Missingness varies substantially across the feature families.

The analysis therefore does not automatically remove rows with missing values or assign business meanings to anonymized variables. More interpretable fields are used for descriptive comparisons, while broader feature handling is left to the modeling notebook.


## 4. Do observable transaction characteristics show different fraud patterns?

Card and email-domain fields provide interpretable transaction context, but the comparisons below remain descriptive. Differences in fraud rate do not establish that these characteristics cause fraud.


In [ ]:
for col in ["card4", "card6"]:
    summary = (
        transactions
        .groupby(col, dropna=False)
        .agg(
            transactions=("TransactionID", "count"),
            fraud_transactions=("isFraud", "sum")
        )
        .reset_index()
    )

    summary["fraud_rate"] = (
        summary["fraud_transactions"]
        / summary["transactions"]
    )

    print(f"\n=== {col} ===")
    display(summary.sort_values("fraud_rate", ascending=False))

In [ ]:
for col in ["P_emaildomain", "R_emaildomain"]:

    summary = (
        transactions
        .groupby(col, dropna=False)
        .agg(
            transactions=("TransactionID", "count"),
            fraud_transactions=("isFraud", "sum")
        )
        .reset_index()
    )

    summary["fraud_rate"] = (
        summary["fraud_transactions"]
        / summary["transactions"]
    )

    summary = summary[
        summary["transactions"] >= 500
    ]

    print(f"\n=== {col} | groups with 500+ transactions ===")

    display(
        summary
        .sort_values("fraud_rate", ascending=False)
        .head(20)
    )

### Finding 4

Observed fraud rates vary across card categories and sufficiently large email-domain groups.

These differences reinforce the idea that fraud exposure is not distributed uniformly across the transaction population, but they should be interpreted as associations rather than causal risk factors.


## 5. How much of the transaction population has identity information?

Identity data is available only for transactions that can be linked to `train_identity.csv`.

I compare identity availability and device type while keeping the partial coverage explicit.


In [ ]:
transactions["HasIdentity"] = (
    transactions["TransactionID"]
    .isin(identity["TransactionID"])
)

identity_presence = (
    transactions
    .groupby("HasIdentity")
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum")
    )
    .reset_index()
)

identity_presence["fraud_rate"] = (
    identity_presence["fraud_transactions"]
    / identity_presence["transactions"]
)

display(identity_presence)

In [ ]:
identity_small = identity[
    ["TransactionID", "DeviceType", "DeviceInfo"]
].copy()

device_audit = transactions[
    ["TransactionID", "isFraud"]
].merge(
    identity_small,
    on="TransactionID",
    how="left"
)

In [ ]:
device_summary = (
    device_audit
    .groupby("DeviceType", dropna=False)
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum")
    )
    .reset_index()
)

device_summary["fraud_rate"] = (
    device_summary["fraud_transactions"]
    / device_summary["transactions"]
)

display(device_summary)

### Finding 5

Identity-linked transactions represent only a subset of the full dataset, and fraud patterns differ between observed identity groups.

Because identity availability is not universal, device-related findings should not be generalized automatically to all transactions.


## 6. Does fraud risk differ across products?

`ProductCD` contains five anonymized product categories. Their business meanings are not disclosed, so the analysis compares the observed codes directly without assigning product names or unsupported explanations.


In [ ]:
transactions["FraudAmount"] = np.where(
    transactions["isFraud"] == 1,
    transactions["TransactionAmt"],
    0
)


In [ ]:
product_identity = (
    transactions
    .groupby(["ProductCD", "HasIdentity"])
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum")
    )
    .reset_index()
)

product_identity["fraud_rate"] = (
    product_identity["fraud_transactions"]
    / product_identity["transactions"]
)

display(product_identity)

In [ ]:
product_summary = (
    transactions
    .groupby("ProductCD")
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum"),
        median_amount=("TransactionAmt", "median"),
        total_value=("TransactionAmt", "sum")
    )
    .reset_index()
)

product_summary["fraud_rate"] = (
    product_summary["fraud_transactions"]
    / product_summary["transactions"]
)

display(
    product_summary.sort_values("fraud_rate", ascending=False)
)

In [ ]:
product_week = (
    transactions
    .groupby(["RelativeWeek", "ProductCD"])
    .agg(
        transactions=("TransactionID", "count"),
        fraud_transactions=("isFraud", "sum")
    )
    .reset_index()
)

product_week["fraud_rate"] = (
    product_week["fraud_transactions"]
    / product_week["transactions"]
)

product_week_pivot = product_week.pivot(
    index="RelativeWeek",
    columns="ProductCD",
    values="fraud_rate"
)

display(product_week_pivot)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

for product in product_week_pivot.columns:
    plt.plot(
        product_week_pivot.index,
        product_week_pivot[product],
        marker="o",
        label=product
    )

plt.xlabel("Relative Week")
plt.ylabel("Fraud Rate")
plt.title("Fraud Rate by Product Over Relative Time")
plt.legend(title="ProductCD")
plt.show()

### Finding 6

Fraud exposure differs across the anonymized product categories and also changes over relative time within those categories.

Identity coverage is not evenly distributed across products, so identity-related differences can be confounded with product mix. Product-level patterns are therefore treated as descriptive signals rather than causal explanations.


## 7. Do fraud frequency and fraud value tell the same story?

A transaction-count fraud rate does not show how much transaction value is associated with fraud.

The final check compares weekly fraud frequency with fraud value exposure.


In [ ]:
transactions["FraudAmount"] = np.where(
    transactions["isFraud"] == 1,
    transactions["TransactionAmt"],
    0
)

weekly_exposure = (
    transactions
    .groupby("RelativeWeek")
    .agg(
        transactions=("TransactionID", "count"),
        total_value=("TransactionAmt", "sum"),
        fraud_transactions=("isFraud", "sum"),
        fraud_value=("FraudAmount", "sum")
    )
    .reset_index()
)

weekly_exposure["fraud_rate"] = (
    weekly_exposure["fraud_transactions"]
    / weekly_exposure["transactions"]
)

weekly_exposure["fraud_value_rate"] = (
    weekly_exposure["fraud_value"]
    / weekly_exposure["total_value"]
)

display(weekly_exposure)

### Finding 7

Fraud transaction rate and fraud value rate are related but not interchangeable.

This distinction matters for downstream control analysis because a policy that changes the number of fraud transactions captured may not affect transaction value exposure in exactly the same way.


## Conclusion

Observed fraud exposure varies across time, product categories, transaction characteristics, identity-linked subsets, and transaction value.

The descriptive audit supports moving beyond a single portfolio-level fraud rate. The next notebook tests whether transaction signals can rank fraud risk and then examines how different risk thresholds trade off fraud capture against legitimate transaction friction.

### What this analysis does not prove

- Product, card, email, identity, or device characteristics cannot be interpreted as causal drivers of fraud.
- `ProductCD` categories cannot be assigned unsupported business meanings.
- `TransactionDT` cannot be mapped to real calendar dates.
- Identity-linked findings cannot be generalized automatically to the full transaction population.
- Fraud transaction value is not the same as verified financial loss.
- The dataset does not contain actual approve, decline, review, or customer-friction outcomes.
